# NumPy & PyTorch Cheatsheet – Beta Guard VAE

Interaktives Lern-Notebook mit genau den NumPy- und PyTorch-Grundoperationen, die in `src/core_real_final/` (`guarded_trainer.py`, `beta_scheduler.py`, `controlvae.py`) und `src/analysis/` vorkommen.

Einfach Zelle für Zelle ausführen und Werte zum Ausprobieren verändern!


In [2]:
import numpy as np
import torch

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)


NumPy: 2.4.6
PyTorch: 2.3.1+cu121


## 1. Shape, `axis` (NumPy) & `dim` (PyTorch)

Bei einer 2D-Matrix `(Zeilen, Spalten)` = `(Samples, Features)`:
- **`0` = Zeilen-Achse (vertikal, Samples)** $\implies$ `.sum(0)` addiert alle Zeilen $\to$ **1 Wert pro Spalte (Feature)**
- **`1` = Spalten-Achse (horizontal, Features)** $\implies$ `.sum(1)` addiert alle Spalten $\to$ **1 Wert pro Zeile (Sample)**
- **`-1` = immer die letzte Achse**


In [3]:
# 2 Patienten (Zeilen) x 3 Gene (Spalten)
x = np.array([[10., 20., 30.],
              [ 2.,  4.,  6.]])
t = torch.tensor(x)

print("Shape:", x.shape, "| ndim:", x.ndim, "| len (Samples):", len(x))
print("Gesamtsumme:         ", x.sum())
print("sum(axis=0) pro Gen: ", x.sum(axis=0))
print("sum(axis=1) pro Pat.:", x.sum(axis=1))
print("PyTorch mean(dim=0): ", t.mean(dim=0))
print("PyTorch mean(dim=1): ", t.mean(dim=1))


Shape: (2, 3) | ndim: 2 | len (Samples): 2
Gesamtsumme:          72.0
sum(axis=0) pro Gen:  [12. 24. 36.]
sum(axis=1) pro Pat.: [60. 12.]
PyTorch mean(dim=0):  tensor([ 6., 12., 18.], dtype=torch.float64)
PyTorch mean(dim=1):  tensor([20.,  4.], dtype=torch.float64)


## 2. `keepdims` / `keepdim` & Broadcasting

**Broadcasting-Regel:** NumPy & PyTorch vergleichen Shapes **von rechts nach links**. Zwei Achsen passen, wenn sie **gleich** sind oder eine davon **`1`** ist.
- NumPy: `keepdims=True` (mit `s`)
- PyTorch: `keepdim=True` (ohne `s`)


In [ ]:
# Zeilen-Mittelwert (pro Patient): braucht keepdims=True!
m_row = x.mean(axis=1, keepdims=True)   # Shape (2, 1)
print("m_row shape:", m_row.shape)
print("x - m_row:\n", x - m_row)        # (2, 3) - (2, 1) -> klappt!

# Spalten-Mittelwert (pro Gen): klappt auch ohne keepdims, weil (3,) von rechts zu (2, 3) passt
m_col = x.mean(axis=0)                  # Shape (3,)
print("\nm_col shape:", m_col.shape)
print("x - m_col:\n", x - m_col)        # (2, 3) - (3,) -> klappt!

# PyTorch-Variante (keepdim ohne s)
print("\nPyTorch t - t.mean(dim=1, keepdim=True):\n", t - t.mean(dim=1, keepdim=True))


## 3. Praxisbeispiel 1: `reconstruction_r2` (`r2_sample` vs. `r2_feature`)

Warum wir beide $R^2$-Arten trennen: Ein **voll kollabierter VAE** gibt für jeden Patienten nur den konstanten Durchschnitts-Patienten `x.mean(0)` aus:
- **`r2_sample`** (AUTOENCODIX) bleibt trotzdem hoch (weil manche Gene immer hoch, andere immer niedrig sind)!
- **`r2_feature`** fällt sofort auf **`0.0`** und entlarvt den Kollaps!


In [ ]:
# 4 Patienten x 3 Gene (Gen 0 niedrig, Gen 1 mittel, Gen 2 hoch)
x_real = np.array([
    [1.0, 5.0, 9.0],
    [1.5, 5.2, 8.8],
    [0.8, 4.8, 9.5],
    [1.2, 5.5, 8.5],
])

# Voll kollabierter Decoder: sagt für jeden Patienten nur den Gen-Mittelwert vorher
recon_collapsed = np.tile(x_real.mean(axis=0), (len(x_real), 1))

sse = (x_real - recon_collapsed) ** 2
sst_per_sample  = ((x_real - x_real.mean(1, keepdims=True)) ** 2).sum(1)
sst_per_feature = ((x_real - x_real.mean(0)) ** 2).sum(0)

r2_sample  = float(np.mean(1 - sse.sum(1) / sst_per_sample))
r2_feature = float(np.mean(1 - sse.sum(0) / sst_per_feature))

print(f"Trotz Total-Kollaps -> r2_sample  = {r2_sample:.4f} (täuschend hoch!)")
print(f"Ehrliches Maß       -> r2_feature = {r2_feature:.4f} (sofort als Kollaps erkannt!)")


## 4. Praxisbeispiel 2: KL-Divergenz, Masken & Lucas-Kollaps ($Q_{99}$)

Genau wie in `guarded_trainer.py`:
- `kl` hat Shape `(N_samples, latent_dim)`
- **ControlVAE-PI:** `kl.sum(dim=1).mean().item()` (erst über Dimensionen summieren, dann über Samples mitteln)
- **Lucas-Kollaps:** Anteil der Samples mit `kl < eps` pro Dimension $\ge 1 - \delta$, äquivalent zu $Q_{99, j} < \epsilon$ über `kthvalue`.


In [ ]:
import math

torch.manual_seed(0)
# 100 Samples x 4 latente Dimensionen
mu     = torch.tensor([0.0, 0.0, 0.8, 1.5]) + 0.01 * torch.randn(100, 4)
logvar = torch.zeros(100, 4)

# Geschlossene KL-Formel (Kingma & Welling 2014 / ControlVAE solver.py)
kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())   # Shape (100, 4)

# 1) ControlVAE Batch-Signal (Skalar)
measured_kl = kl.sum(dim=1).mean().item()
print(f"ControlVAE total_KL (Skalar): {measured_kl:.4f} nats")

# 2) Epochen-Mittelwert pro Dimension (sum(dim=0) / N)
rates_per_dim = (kl.sum(dim=0, dtype=torch.float64) / kl.shape[0])
print("Mittlere KL pro Dimension:   ", [round(v, 4) for v in rates_per_dim.tolist()])
print("Summe der 4 Dimensionen:     ", round(rates_per_dim.sum().item(), 4), "(== total_KL!)")

# 3) Lucas (eps=0.01, delta=0.01)
eps, delta = 0.01, 0.01
counts    = (kl < eps).sum(dim=0)
fractions = counts.double() / kl.shape[0]
collapsed = fractions >= (1 - delta)
q99       = kl.kthvalue(math.ceil((1 - delta) * kl.shape[0]), dim=0).values

print("Fractions < eps:  ", [round(v, 3) for v in fractions.tolist()])
print("Q99 pro Dimension:", [round(v, 4) for v in q99.tolist()])
print("Kollabierte Dims: ", torch.where(collapsed)[0].tolist())


## 5. `torch.cat` vs. `torch.stack` & PyTorch $\leftrightarrow$ NumPy

- **`torch.cat(..., dim=0)`**: hängt Batches an bestehende Achse an (`(4, 8) + (4, 8) -> (8, 8)`)
- **`torch.stack(..., dim=0)`**: erzeugt eine **neue** Achse (`10x (4, 8) -> (10, 4, 8)`, z. B. für 10 Monte-Carlo-Draws)
- **PyTorch $\to$ NumPy:** immer `.detach().cpu().numpy()` (vom Autograd-Graphen lösen $\to$ auf CPU $\to$ NumPy)


In [ ]:
b1 = torch.zeros(4, 8)
b2 = torch.ones(4, 8)

print("cat(dim=0):  ", torch.cat([b1, b2], dim=0).shape)
print("stack(dim=0):", torch.stack([b1, b2], dim=0).shape)

# 10 Monte-Carlo-Draws mitteln (wie in _export_posteriors)
mc_draws = [torch.randn(4, 8) for _ in range(10)]
mc_mean = torch.stack(mc_draws, dim=0).mean(dim=0)   # (10, 4, 8) -> (4, 8)
print("MC-Mean Shape:", mc_mean.shape)

# Umwandlung Tensor <-> NumPy & Skalar
t_grad = torch.tensor([[1.0, 2.0]], requires_grad=True)
arr = t_grad.detach().cpu().numpy()
print("NumPy:", arr, type(arr), "| Skalar via .item():", t_grad.sum().item())


## 6. `model.train()` vs. `model.eval()`, `no_grad` & `with`-Context-Manager

- **`model.train()`**: Dropout AN, BatchNorm nutzt Batch-Statistik (deshalb entsteht der Train/Eval-Gap!)
- **`model.eval()`**: Dropout AUS, BatchNorm nutzt laufende Statistik
- **`with torch.inference_mode():`** / **`with torch.no_grad():`**: schaltet Gradienten nur für den eingerückten Block aus und danach automatisch wieder ein.


In [ ]:
layer = torch.nn.Dropout(p=0.5)
inp = torch.ones(1, 8)

layer.train()
print("train() (Dropout AN): ", layer(inp))

layer.eval()
with torch.inference_mode():
    out = layer(inp)
print("eval()  (Dropout AUS):", out, "| requires_grad:", out.requires_grad)
